# Notebook 02: Converting FHIR Data to Cascade Protocol

This notebook demonstrates how to convert FHIR JSON resources to Cascade Protocol Turtle format using the Python SDK.

In clinical workflows, patient data often arrives as FHIR R4 JSON from EHR exports or health apps. The Cascade Protocol sits above FHIR as a provenance-annotated, privacy-preserving serialization layer.

**Covered topics:**
1. Mapping FHIR MedicationRequest to Cascade `Medication`
2. Mapping FHIR Condition to Cascade `Condition`
3. Mapping FHIR Observation to Cascade `VitalSign`
4. Serializing to Turtle with proper provenance annotation
5. Validating the output

**Setup:** `pip install cascade-protocol`

In [ ]:
import sys
from pathlib import Path

sdk_root = Path("../src").resolve()
if str(sdk_root) not in sys.path:
    sys.path.insert(0, str(sdk_root))

from cascade_protocol import (
    Medication, Condition, VitalSign, LabResult,
    serialize, validate, CURRENT_SCHEMA_VERSION
)
print(f"SDK ready. Schema version: {CURRENT_SCHEMA_VERSION}")

## Simulated FHIR Data

In production, this data comes from an EHR system. Here we use synthetic examples.

In [ ]:
# Simulated FHIR R4 MedicationRequest
fhir_medication_request = {
    "resourceType": "MedicationRequest",
    "id": "med-fhir-001",
    "status": "active",
    "medicationCodeableConcept": {
        "coding": [
            {"system": "http://www.nlm.nih.gov/research/umls/rxnorm", "code": "197884", "display": "Lisinopril 20 MG Oral Tablet"}
        ],
        "text": "Lisinopril"
    },
    "dosageInstruction": [
        {
            "text": "Take once daily",
            "doseAndRate": [{"doseQuantity": {"value": 20, "unit": "mg"}}],
            "route": {"coding": [{"display": "oral"}]}
        }
    ],
    "requester": {"display": "Dr. Sarah Chen"},
    "authoredOn": "2024-06-15"
}

## 1. Map FHIR MedicationRequest → Cascade Medication

In [ ]:
def fhir_medication_to_cascade(fhir_med: dict) -> Medication:
    """Convert a FHIR R4 MedicationRequest to a Cascade Protocol Medication."""
    import uuid
    
    # Extract the medication name
    name = fhir_med.get("medicationCodeableConcept", {}).get("text", "Unknown")
    
    # Extract dose from dosageInstruction
    dose = None
    route = None
    dosage = fhir_med.get("dosageInstruction", [])
    if dosage:
        d = dosage[0]
        dose_qty = d.get("doseAndRate", [{}])[0].get("doseQuantity", {})
        if dose_qty:
            dose = f"{dose_qty.get('value', '')} {dose_qty.get('unit', '')}".strip()
        route_coding = d.get("route", {}).get("coding", [{}])
        if route_coding:
            route = route_coding[0].get("display")
    
    # Extract RxNorm code
    rx_norm_code = None
    for coding in fhir_med.get("medicationCodeableConcept", {}).get("coding", []):
        if "rxnorm" in coding.get("system", "").lower():
            rx_norm_code = f"{coding['system']}/{coding['code']}"
    
    return Medication(
        id=f"urn:uuid:{uuid.uuid4()}",
        medication_name=name,
        is_active=fhir_med.get("status") == "active",
        dose=dose,
        route=route,
        prescriber=fhir_med.get("requester", {}).get("display"),
        start_date=fhir_med.get("authoredOn", "") + "T00:00:00Z" if fhir_med.get("authoredOn") else None,
        rx_norm_code=rx_norm_code,
        source_fhir_resource_type="MedicationRequest",
        provenance_class="healthKitFHIR",
        data_provenance="ClinicalGenerated",
        schema_version=CURRENT_SCHEMA_VERSION,
    )

cascade_med = fhir_medication_to_cascade(fhir_medication_request)
print(f"Medication: {cascade_med.medication_name}")
print(f"Dose: {cascade_med.dose}")
print(f"Route: {cascade_med.route}")
print(f"Prescriber: {cascade_med.prescriber}")
print(f"Provenance: {cascade_med.data_provenance} / {cascade_med.provenance_class}")

## 2. Serialize to Turtle

In [ ]:
turtle = serialize(cascade_med)
print(turtle)

## 3. Validate the Output

In [ ]:
result = validate(turtle)
print(f"Valid: {result.is_valid}")
if result.errors:
    print(f"Errors: {result.errors}")
if result.warnings:
    print(f"Warnings: {result.warnings}")

## 4. Multiple Records from FHIR Bundle

In [ ]:
# Simulated FHIR Condition resources
fhir_conditions = [
    {"code": {"text": "Essential Hypertension"}, "clinicalStatus": {"coding": [{"code": "active"}]}, "onsetDateTime": "2019-03-01"},
    {"code": {"text": "Type 2 Diabetes Mellitus"}, "clinicalStatus": {"coding": [{"code": "active"}]}, "onsetDateTime": "2020-11-15"},
    {"code": {"text": "Asthma"}, "clinicalStatus": {"coding": [{"code": "active"}]}, "onsetDateTime": "2005-06-20"},
]

import uuid

def fhir_condition_to_cascade(fhir_cond: dict) -> Condition:
    name = fhir_cond.get("code", {}).get("text", "Unknown")
    status_coding = fhir_cond.get("clinicalStatus", {}).get("coding", [{}])
    status = status_coding[0].get("code", "active") if status_coding else "active"
    onset = fhir_cond.get("onsetDateTime")
    if onset:
        onset = onset + "T00:00:00Z"
    return Condition(
        id=f"urn:uuid:{uuid.uuid4()}",
        condition_name=name,
        status=status,
        onset_date=onset,
        data_provenance="ClinicalGenerated",
        schema_version=CURRENT_SCHEMA_VERSION,
    )

cascade_conditions = [fhir_condition_to_cascade(c) for c in fhir_conditions]

print("Converted conditions:")
for cond in cascade_conditions:
    print(f"  {cond.condition_name} ({cond.status}) — onset: {cond.onset_date}")

In [ ]:
# Serialize and display the first condition
turtle = serialize(cascade_conditions[0])
print(turtle)

## 5. Batch Validation

In [ ]:
from cascade_protocol.pandas_integration import records_to_dataframe

all_conditions = cascade_conditions

# Validate all records
for cond in all_conditions:
    result = validate(cond)
    status = "PASS" if result.is_valid else f"FAIL: {result.errors}"
    print(f"  {cond.condition_name}: {status}")

# Convert to DataFrame
df = records_to_dataframe(all_conditions)
print(f"\nDataFrame with {len(df)} conditions:")
df[["condition_name", "status", "onset_date"]].head()